# Clase 8 — Práctica guiada: correr un LLM local con llama.cpp

> En esta clase sí vamos a tocar una herramienta real, pero con la menor complejidad posible. La meta no es entender toda la ingeniería interna, sino ver un LLM local funcionando y aprender a probarlo con criterio.

## Objetivos de hoy

| Paso | Qué vamos a hacer |
|---|---|
| 1 | Verificar dependencias |
| 2 | Descargar un modelo GGUF pequeño |
| 3 | Cargarlo con llama.cpp |
| 4 | Hacer una primera consulta |
| 5 | Probar temperatura, longitud de salida y calidad de prompt |
| 6 | Cerrar con una actividad práctica |

---
## 1. Qué necesitás antes de empezar

Este notebook usa dos librerías principales:

- huggingface_hub: para descargar el archivo del modelo.
- llama_cpp: para cargar y ejecutar el modelo local.

Si te falta alguna, instalala desde terminal:

```bash
pip install huggingface-hub llama-cpp-python
```

En Apple Silicon puede hacer falta reinstalar llama-cpp-python con soporte Metal si querés aceleración. Para esta clase vamos a asumir un arranque simple, pensado para CPU.

In [1]:
import os
import time

try:
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
except ImportError as error:
    raise ImportError(
        "Faltan dependencias. Instalá: pip install huggingface-hub llama-cpp-python"
    ) from error

print("Dependencias listas.")

Dependencias listas.


---
## 2. Elegir un modelo realista para clase

Para una clase inicial conviene usar un modelo pequeño. Vamos a usar una versión cuantizada en formato GGUF. Eso lo hace mucho más liviano que el modelo original.

### Idea práctica

- Un modelo pequeño arranca más rápido.
- Un archivo GGUF cuantizado ocupa menos memoria.
- El resultado no va a ser perfecto, pero alcanza para experimentar.

### Elegir el modelo
- Qwen 2.5 
- 0.5B 
- Q4 
- GGUF

Es un modelo de 500 millones de parámetros, cuantizado a 4 bits, y pesa alrededor de 500 MB en formato GGUF.

In [2]:
REPO_ID = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

print("Modelo elegido:")
print("- Repositorio:", REPO_ID)
print("- Archivo:", FILENAME)

Modelo elegido:
- Repositorio: Qwen/Qwen2.5-0.5B-Instruct-GGUF
- Archivo: qwen2.5-0.5b-instruct-q4_k_m.gguf


In [3]:
inicio_descarga = time.time()
ruta_modelo = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
tiempo_descarga = time.time() - inicio_descarga
tamano_mb = os.path.getsize(ruta_modelo) / (1024 * 1024)

print("Modelo listo.")
print("- Ruta local:", ruta_modelo)
print(f"- Tamaño: {tamano_mb:.1f} MB")
print(f"- Tiempo de preparación: {tiempo_descarga:.1f} s")

Modelo listo.
- Ruta local: /Users/inti/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf
- Tamaño: 468.6 MB
- Tiempo de preparación: 0.4 s


---
## 3. Cargar el modelo con llama.cpp

En esta versión vamos a priorizar compatibilidad. Por eso arrancamos con n_gpu_layers = 0, es decir, solo CPU. Más adelante podés probar aceleración local si tu máquina y tu instalación lo soportan.

In [4]:
inicio_carga = time.time()
llm = Llama(
    model_path=ruta_modelo,
    n_ctx=2048,
    n_gpu_layers=0,
    verbose=False
)
tiempo_carga = time.time() - inicio_carga

print("Modelo cargado en memoria.")
print(f"- Tiempo de carga: {tiempo_carga:.1f} s")
print("- Contexto máximo:", llm.n_ctx())
print("- Tamaño de vocabulario:", llm.n_vocab())

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Modelo cargado en memoria.
- Tiempo de carga: 0.6 s
- Contexto máximo: 2048
- Tamaño de vocabulario: 151936


---
## 4. Una función simple para hacer preguntas

Para no repetir código, armamos una función pequeña. El punto importante es este: el modelo no solo necesita una pregunta. También suele ayudar darle un rol o una instrucción breve.

In [5]:
def preguntar(
    pregunta,
    system_prompt="Sos un profesor paciente. Explicá de forma breve y clara.",
    temperature=0.7,
    max_tokens=120
):
    respuesta = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": pregunta}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return respuesta["choices"][0]["message"]["content"].strip()

print("Función lista.")

Función lista.


---
## 5. Primera consulta

Empecemos con una consigna corta y fácil de evaluar.

In [6]:
pregunta = "Explicá en dos frases qué es un token en un modelo de lenguaje."
respuesta = preguntar(pregunta)

print("Pregunta:")
print(pregunta)
print()
print("Respuesta:")
print(respuesta)

Pregunta:
Explicá en dos frases qué es un token en un modelo de lenguaje.

Respuesta:
Un token es un elemento fundamental en un modelo de lenguaje, como el de una inteligencia artificial, que se utiliza para representar una palabra, un número, una letra o cualquier otro tipo de dato. Es el símbolo que se utiliza para describir el contenido de un dato en la representación del lenguaje, y se utilizan en el procesamiento de la información para que pueda interpretarse y procesarse por el modelo de lenguaje.


---
## 6. Experimento A: cambiar la temperatura

Vamos a usar la misma pregunta varias veces. La idea es ver cómo cambia el estilo de la respuesta.

In [7]:
pregunta_temp = "Dame una analogia simple para entender que hace un LLM."

for temperatura in [0.2, 0.8, 1.3]:
    print(f"Temperatura = {temperatura}")
    print("-" * 60)
    print(preguntar(pregunta_temp, temperature=temperatura, max_tokens=100))
    print()

Temperatura = 0.2
------------------------------------------------------------
Un LLM (Large Language Model) es como un maestro de la lengua que puede hablar en muchas palabras y formas diferentes, pero siempre con el mismo propósito: enseñar y explicar. Es como si tu maestro de la lengua tuviera la capacidad de hablar en todas las línguas y formas de expresión, pero siempre con el mismo sentido y objetivo.

Temperatura = 0.8
------------------------------------------------------------
Un LLM (Language Model) es como un jefe de inteligencia artificial en el mundo real. Es capaz de entender y procesar lenguaje, aprender y adaptarse a nuevas situaciones y datos. Es como un jefe que sabe leer y escribir, y puede entender y responder a la pregunta de una persona.

Temperatura = 1.3
------------------------------------------------------------
Un LLM es como un gran detective. No hay lugar más preciso. Se acerca al caso, enseguida observa sus signos y señales, analiza lo que se puede ver y o

> Pregunta para discutir: ¿en cuál de las tres respuestas sentís más claridad? ¿y en cuál aparece más variación o riesgo de divagar?

---
## 7. Experimento B: prompt vago contra prompt claro

Muchos problemas que atribuimos al modelo en realidad vienen de una instrucción poco precisa.

In [8]:
prompt_vago = "Habla sobre inteligencia artificial."
prompt_claro = "Explicá qué es la inteligencia artificial para estudiantes de secundaria en 4 bullets, sin tecnicismos."

print("Prompt vago")
print("-" * 60)
print(preguntar(prompt_vago, max_tokens=120))
print()
print("Prompt claro")
print("-" * 60)
print(preguntar(prompt_claro, max_tokens=120))

Prompt vago
------------------------------------------------------------
Soy un profesor paciente, por lo tanto, no tengo la capacidad de actuar como un profesor paciente. Sin embargo, puedo ayudarte a entender y explicar la inteligencia artificial de la mejor manera posible, si lo deseas.

Prompt claro
------------------------------------------------------------
1. La IA en secundaria: Es una inteligencia artificial que se utiliza en el aprendizaje automatizado y la generación de texto para ayudar a los estudiantes de secundaria en sus tareas de estudio.

2. Inversión en IA en secundaria: Se mantiene una inversión en la IA para que los estudiantes se sientan preparados para las nuevas tecnologías y métodos de aprendizaje.

3. Beneficios de la IA en secundaria: La IA puede ayudar a los estudiantes a resolver problemas más complejos y a


---
## 8. Experimento C: longitud máxima de salida

Otro parámetro importante es max_tokens. No cambia el conocimiento del modelo, pero sí cuánto lo dejamos hablar.

In [9]:
consigna = "Resumí en lenguaje simple por qué llama.cpp es útil para probar LLMs locales."

for limite in [40, 120]:
    print(f"max_tokens = {limite}")
    print("-" * 60)
    print(preguntar(consigna, max_tokens=limite))
    print()

max_tokens = 40
------------------------------------------------------------
Cuando probar LLMs locales, llam.cpp es útil porque es una herramienta que te permite hacer pruebas rápidas y fácilmente. Te ayuda a verificar si las cosas funcionan

max_tokens = 120
------------------------------------------------------------
Entiendo que quieres que me explique rápido y claro cómo un profesor paciente puede entender la idea de "llam.cpp" en el contexto de probar LLMs locales. 

"LLM" significa "Language Model", es un lenguaje de inteligencia artificial que se encarga de entender y pronunciarse en una lengua específica. 

"Local" se refiere a que el lenguaje de IA se basa en una lengua específica en un lugar particular (el profesor). 

Por lo tanto, "llam.cpp" es útil para probar L



---
## 9. Actividad guiada

Probá estas consignas y compará las respuestas:

- Explicá qué es la temperatura de un modelo con una analogía cotidiana.
- Decime tres diferencias entre usar una API y usar un modelo local.
- Enseñame con un ejemplo simple por qué el contexto importa en un LLM.

La idea de la actividad no es conseguir una respuesta perfecta, sino aprender a observar el efecto de cada parámetro y de cada prompt.

In [ ]:
prompts_practica = [
    "Explicá qué es la temperatura de un modelo con una analogía cotidiana.",
    "Decime tres diferencias entre usar una API y usar un modelo local.",
    "Enseñame con un ejemplo simple por qué el contexto importa en un LLM."
]

for prompt in prompts_practica:
    print("Prompt:", prompt)
    print("-" * 60)
    print(preguntar(prompt, temperature=0.7, max_tokens=120))
    print("=" * 60)
    print()